# DFS Weekly Pipeline

> **DEV — never validated against a live DraftKings slate.** The 2025 season came and
> went without an end-to-end live test, and the only salary file in the repo is still
> `dk_salaries_2025_week10_synthetic.csv`. Treat all outputs as experimental until a
> real slate has been run end-to-end and logged.

**Run this notebook each week** to generate an optimised DraftKings NFL Classic lineup.

### Workflow
1. Run `predict_fantasy.ipynb` for the target week (or let GitHub Actions do it).
2. When the DK slate posts, export the salary CSV from any NFL Classic contest lobby
   (*Available Players → Export to CSV*) and note its path.
3. Set `CSV_PATH` (and optionally `SEASON` / `WEEK`) in the Parameters cell below.
4. Run all cells top-to-bottom. Review the player pool, adjust `LOCKED` / `EXCLUDED`,
   and re-run the optimizer cell as needed.

### DraftKings NFL Classic roster
| Slot | Count |
|---|---|
| QB | 1 |
| RB | 2 |
| WR | 3 |
| TE | 1 |
| FLEX (RB/WR/TE) | 1 |
| DST | 1 |
| **Total** | **9** |

Salary cap: **$50,000**. Max 8 players from the same team.


## Parameters

Set `CSV_PATH` to the DK salary export for this week's slate.
`SEASON` and `WEEK` default to `None`, which auto-detects the most recent projection
file in `fantasy/fantasy_projections/` — override them if you want a specific week.

`LOCKED` / `EXCLUDED` accept player names exactly as they appear in the DK salary CSV.


In [11]:
# Parameters (papermill-injectable)
# Path to the DK salary CSV exported from the contest lobby
CSV_PATH = "dk_salaries.csv"
# None = auto-detect from the most recent projection file
SEASON = None
WEEK = None
BUDGET = 50_000
LOCKED = []
EXCLUDED = []


## Setup

Data loading is defined here. The two pieces of real logic are imported, not redefined:
the matching cascade + DK scoring from `dfs_matching.py`, and the ILP from
`lineup_optimizer.py`. Both used to be copy-pasted into this notebook AND
`optimizer.ipynb`.


In [ ]:
import io
import re
import sys
from pathlib import Path

import pandas as pd
import pulp

# ── Path resolution ────────────────────────────────────────────────────────
_HERE = Path().resolve()
_CANDIDATE_DIRS = [
    _HERE, _HERE / "fantasy" / "dfs", _HERE.parent, _HERE.parent / "dfs",
    _HERE.parent.parent / "fantasy" / "dfs",
]
DFS_DIR = next((p for p in _CANDIDATE_DIRS if (p / "dfs_matching.py").exists()), None)
if DFS_DIR is None:
    raise RuntimeError("Could not find dfs_matching.py (expected in fantasy/dfs/).")
if str(DFS_DIR) not in sys.path:
    sys.path.insert(0, str(DFS_DIR))

PROJ_DIR = next(
    (p for p in [
        _HERE / "fantasy_projections",
        _HERE / "fantasy" / "fantasy_projections",
        _HERE.parent / "fantasy_projections",
        _HERE.parent / "fantasy" / "fantasy_projections",
        _HERE.parent.parent / "fantasy" / "fantasy_projections",
    ] if p.exists()),
    _HERE / "fantasy_projections",
)
if not PROJ_DIR.exists():
    raise RuntimeError(f"Could not find fantasy_projections directory. PROJ_DIR={PROJ_DIR}")

# ── Matching cascade + DK scoring live in dfs_matching.py ───────────────────
# They used to be copy-pasted here AND in optimizer.ipynb, and both copies matched
# DK names with a bare difflib cutoff of 0.72 over EVERY projection name — no position
# and no team constraint. Leave-one-out over projections_2025_week10.csv: 150 of 568
# absent players still got a "match", 95 of them CROSS-POSITION (Josh Allen QB ->
# Josh Palmer WR, Aaron Rodgers QB -> Aaron Jones RB, ...), every one labelled
# `match == "model"` so the unmatched warning below never saw them. The shared module
# replaces that with an id -> exact(position+team) -> alias -> constrained-fuzzy
# cascade whose leave-one-out false-match rate is 0/568. See dfs_matching.py.
from dfs_matching import (            # noqa: E402
    ALL_STATUSES, MODEL_STATUSES, STAT_COLS,
    assert_objective_finite, calc_dk_proj_pts, format_match_report,
    match_status_counts, merge_projections, norm_name, norm_team, num, pos_class,
)

_DK_COL_MAP = {
    "Position":         "position",
    "Name":             "name",
    "Salary":           "salary",
    "TeamAbbrev":       "team",
    "AvgPointsPerGame": "avg_pts",
    "Game Info":        "game_info",
}

# Optional stable-id columns on the DK side. DK's own "ID" is a DraftKings id, NOT a
# gsis id, so it is deliberately NOT mapped here — only a real nflverse/gsis id column
# (player_id / gsis_id / nfl_id) enables step 1 of the cascade.
_DK_ID_COL_MAP = {"gsis_id": "player_id", "nfl_id": "player_id", "player_id": "player_id"}


def available_weeks():
    files = sorted(PROJ_DIR.glob("projections_*_week*.csv"), reverse=True)
    out = []
    for f in files:
        m = re.match(r"projections_(\d{4})_week(\d{2})\.csv", f.name)
        if m:
            out.append((int(m.group(1)), int(m.group(2))))
    return out


def load_projections(season, week):
    path = PROJ_DIR / f"projections_{season}_week{week:02d}.csv"
    if not path.exists():
        raise FileNotFoundError(f"No projection file: {path}")
    return pd.read_csv(path)


def load_dk_salaries(path_or_bytes):
    raw = open(path_or_bytes, "rb").read() if isinstance(path_or_bytes, (str, Path)) else path_or_bytes
    df  = pd.read_csv(io.BytesIO(raw))
    df  = df.rename(columns={k: v for k, v in _DK_COL_MAP.items() if k in df.columns})
    df  = df.rename(columns={k: v for k, v in _DK_ID_COL_MAP.items() if k in df.columns})
    df["salary"]  = pd.to_numeric(df["salary"].astype(str).str.replace("$","",regex=False).str.replace(",","",regex=False), errors="coerce")
    df["avg_pts"] = pd.to_numeric(df.get("avg_pts", 0), errors="coerce").fillna(0)
    df = df.dropna(subset=["salary", "name", "position"])
    keep = ["position","name","salary","team","avg_pts","game_info","player_id"]
    return df[[c for c in keep if c in df.columns]].copy()

# ── The ILP lives in lineup_optimizer.py ─────────────────────────────
# optimize_lineup / _assign_slots used to be copy-pasted here AND in optimizer.ipynb,
# the same duplication that let the matcher defect live in two places. The solver is
# PuLP's bundled CBC binary -- assert_cbc_available() records exactly which one, and
# raises if it is absent rather than falling back to some other solver.
from lineup_optimizer import (            # noqa: E402
    assert_cbc_available, assign_slots, lineup_facts, optimize_lineup,
)

_SOLVER = assert_cbc_available()

print("All functions loaded (matching cascade from dfs_matching.py).")
print(f"Projections directory: {PROJ_DIR}  (exists: {PROJ_DIR.exists()})")
print(f"Match statuses: {', '.join(ALL_STATUSES)}")
print(f"ILP solver: {_SOLVER['solver_name']} (CBC {_SOLVER['cbc_version']}) at {_SOLVER['path']}")


## Step 1 — Load Model Projections

Our XGBoost models produce `projected_pts` (half-PPR) for each skill-position player.
We use these rather than DK's season average because they incorporate:
- Current injury status and practice participation
- Depth chart position (starter vs. backup)
- Rolling offensive/defensive EPA over the last 4 games
- Matchup difficulty (opponent defensive EPA allowed)

If `SEASON` / `WEEK` are `None`, we auto-detect the most recent projection file.


In [13]:
weeks = available_weeks()
if not weeks:
    raise RuntimeError(f"No projection files found in {PROJ_DIR}. Run predict_fantasy.ipynb first.")

if SEASON is None or WEEK is None:
    SEASON, WEEK = weeks[0]

print(f"Using projections: Season {SEASON}, Week {WEEK}")
proj_df = load_projections(SEASON, WEEK)
print(f"  {len(proj_df)} players loaded")
proj_df[["player_display_name","position","team","projected_pts"]].head(10)


Using projections: Season 2025, Week 10
  568 players loaded


,player_display_name,position,team,projected_pts
0,Brock Purdy,QB,SF,24.06
1,Josh Allen,QB,BUF,22.34
2,Bo Nix,QB,DEN,21.05
3,Drake Maye,QB,NE,19.78
4,Jared Goff,QB,DET,18.97
5,Lamar Jackson,QB,BAL,18.97
6,Jalen Hurts,QB,PHI,18.71
7,Kyler Murray,QB,ARI,18.67
8,Trevor Lawrence,QB,JAX,18.63
9,Jordan Love,QB,GB,18.39


## Step 2 — Load DraftKings Salary Data

Set `CSV_PATH` in the Parameters cell to the file you exported from the DK contest
lobby (*Available Players → Export to CSV*). Expected columns:
`Position`, `Name`, `Salary`, `TeamAbbrev`, `AvgPointsPerGame`.

The file is available once DK posts the slate for the week (typically Thursday
for the main Sunday slate, earlier for Thursday Night Football).

**DST rows** use DK's season `AvgPointsPerGame` — no team-defense projection model yet.


In [14]:
dk_df = load_dk_salaries(CSV_PATH)
print(f"DK salary file : {CSV_PATH}")
print(f"  {len(dk_df)} players | positions: {dk_df['position'].value_counts().to_dict()}")
dk_df.sort_values("salary", ascending=False).head(10)


Fetching DK salaries from RotoGuru (nfldfs): Season 2025, Week 10 ...


RuntimeError: RotoGuru returned no player data for 2025 week 10. This week may not be indexed yet, or the site format has changed.

## Step 3 — Build the Player Pool

`merge_projections()` links each DK player to our projection through a **constrained
matching cascade** (`dfs_matching.py`) and pulls through the per-stat columns needed for
DK scoring. The cascade replaced a bare `difflib` fuzzy match with no position or team
constraint, which on the week-10 file substituted a *different* player for 150 of 568
absent players — 95 of them at a different position — and labelled every one `model`.

**Cascade order (first hit wins):**

| Status | Meaning |
|---|---|
| `id` | stable player id present on both sides and equal |
| `exact` | normalized name equal **and** compatible position **and** same team |
| `team_mismatch` | name+position unique league-wide but the teams disagree (trade / stale team). Projection IS used; flagged for review |
| `alias` | small hand-reviewed spelling table (e.g. Josh ↔ Joshua Palmer), still position-constrained |
| `fuzzy` | same team **and** same position only, needs both a high score and a margin over the runner-up |
| `ambiguous` | two candidates survived, or the fuzzy leader missed the margin — **never guessed**; uses the DK average |
| `unmatched` | no candidate; uses the DK average |
| `dst` | team defense; there is no DST projection model |

There is deliberately **no generic `model` status** any more — ask `used_model` (or
`MODEL_STATUSES`) for "did this row use our projection?".

`calc_dk_proj_pts()` converts the half-PPR model output to **DraftKings Classic** points:

| Adjustment | Positions | Detail |
|---|---|---|
| +0.5 pts/reception | WR, TE | Full PPR (DK=1.0) vs half-PPR (our model=0.5) |
| +0.5 × est. receptions | RB | Rec yards ÷ 7 yds/rec estimate |
| +3 × P(300+ pass yds) | QB | Milestone bonus, Normal-approximated |
| +3 × P(100+ rush yds) | RB | Milestone bonus, Normal-approximated |
| +3 × P(100+ rec yds) | WR, TE | Milestone bonus, Normal-approximated |

Every stat read goes through `num()`, which maps `None`/NaN to 0. The old code used
`float(x or 0)`, which does **not** catch NaN (NaN is truthy), so an unmatched row's NaN
reached the pulp objective coefficient. `assert_objective_finite()` now fails loudly
before the solve rather than letting a NaN into the ILP.

The **value** column (`dfs_proj_pts / salary_in_$k`) ranks players by DK efficiency.


In [ ]:
players = merge_projections(dk_df, proj_df)
players["dfs_proj_pts"] = calc_dk_proj_pts(players)
players["value"] = (players["dfs_proj_pts"] / (players["salary"] / 1000)).round(2)

# Surface the FULL match breakdown before anything is optimized. Anything that is not
# an id/exact/team_mismatch/alias/fuzzy hit is using DK's season average, not our model.
print(format_match_report(players))

counts = match_status_counts(players)
n_fallback = counts["ambiguous"] + counts["unmatched"]
if n_fallback:
    print(f"\n⚠  {n_fallback} skill-position row(s) are NOT on a model projection.")
else:
    print("\nAll skill-position players resolved to a model projection ✅")
if counts["team_mismatch"]:
    print(f"ℹ  {counts['team_mismatch']} row(s) matched by name+position with a TEAM "
          f"disagreement (trade or stale team) — check them.")
if counts["fuzzy"]:
    print(f"ℹ  {counts['fuzzy']} row(s) resolved by same-team/same-position fuzzy match.")

# No NaN may reach the ILP objective.
assert_objective_finite(players, "dfs_proj_pts")

# Spot-check: DK pts vs raw half-PPR for top modelled players
sample = players[players["used_model"]].nlargest(5, "dfs_proj_pts")[
    ["position","name","match","proj_pts","dfs_proj_pts"]
].copy()
sample["dk_uplift"] = (sample["dfs_proj_pts"] - sample["proj_pts"]).round(2)
print("\nDK scoring uplift — top 5 modelled players:")
print(sample.to_string(index=False))


In [ ]:
# Full player pool — sorted by DK value within each position
pool = (
    players.sort_values(["position","value"], ascending=[True, False])
    [["position","name","team","salary","proj_pts","dfs_proj_pts","value","match","match_name"]]
)
print(f"Total player pool: {len(pool)} players")
pool


### Player Pool Analysis

Before optimising it's useful to review the landscape:
- Which positions are salary-compressed (everyone priced similarly)?
- Are there clear value outliers — players priced low relative to projection?
- Does the unmatched list include any must-starts worth manually adjusting?

The position breakdown and top-value lists below answer these quickly.


In [ ]:
print("=== Position breakdown ===")
print(players.groupby("position")[["salary","proj_pts","value"]].agg(["mean","max","min"]).round(1).to_string())

print("\n=== Top 5 by value at each position ===")
for pos in ["QB","RB","WR","TE","DST"]:
    sub = players[players["position"]==pos].nlargest(5,"value")[["name","team","salary","proj_pts","value"]]
    print(f"\n{pos}")
    print(sub.to_string(index=False))


## Step 4 — Optimize the Lineup

The ILP solver maximises total `dfs_proj_pts` (DK Classic full-PPR + milestone bonuses)
subject to the DK Classic roster rules and salary cap. The FLEX slot is filled
automatically — you don't specify whether it should be a RB, WR, or TE.

Modify `LOCKED` / `EXCLUDED` in the Parameters cell and re-run this cell to explore
alternative builds.


In [ ]:
print(f"Running optimizer  budget=${BUDGET:,}  locked={LOCKED}  excluded={EXCLUDED} ...")
lineup = optimize_lineup(players, budget=BUDGET, locked=LOCKED, excluded=EXCLUDED,
                         objective_col="dfs_proj_pts")

if lineup is None:
    print("\n❌  No feasible lineup found. Possible causes:")
    print("   - Not enough players at a position in the salary CSV")
    print("   - Lock constraints are contradictory or exceed the cap")
    print("   - Exclusions remove too many players at a position")
else:
    print("Optimal lineup found ✅")
    # Every DK Classic rule, MEASURED off the returned lineup (not printed and trusted).
    _f = lineup_facts(lineup, "dfs_proj_pts")
    assert _f["lp_status"] == "Optimal", _f
    assert _f["n_players"] == 9, _f
    assert _f["salary"] <= BUDGET, _f
    assert _f["pos_counts"]["QB"] == 1 and _f["pos_counts"]["DST"] == 1, _f
    assert _f["pos_counts"]["RB"] >= 2 and _f["pos_counts"]["WR"] >= 3 and _f["pos_counts"]["TE"] >= 1, _f
    assert _f["flex_bodies"] == 7, _f          # FLEX legality
    assert _f["max_from_one_team"] <= 8, _f
    print(f"  constraints verified: {_f}")


## Results

The table below shows the optimal 9-player roster with DK slot labels.
`FLEX` indicates the extra skill-position player chosen by the solver.

Review:
- **Total salary** — ideally within $500 of the cap (unused cap = lost points).
- **Salary distribution** — are you paying up at the right positions?
- **Match quality** — any FLEX or anchor picks sourced from `dk_avg`? If so, check the
  actual player's status before locking the lineup.


In [ ]:
if lineup is not None:
    total_sal  = lineup["salary"].sum()
    total_pts  = lineup["dfs_proj_pts"].sum()
    remaining  = BUDGET - total_sal

    print(f"Projected DK pts : {total_pts:.1f}  (half-PPR base: {lineup['proj_pts'].sum():.1f})")
    print(f"Total salary     : ${total_sal:,.0f}")
    print(f"Remaining cap    : ${remaining:,.0f}")
    print()

    display_cols = ["Slot","name","team","salary","proj_pts","dfs_proj_pts","value","match"]
    print(lineup[display_cols].to_string(index=False))


In [ ]:
# Salary allocation by position
if lineup is not None:
    sal_by_pos = lineup.groupby("position")["salary"].sum().sort_values(ascending=False)
    pct        = (sal_by_pos / total_sal * 100).round(1)
    print("=== Salary allocation ===")
    for pos, sal in sal_by_pos.items():
        print(f"  {pos:<4}  ${sal:>6,.0f}  ({pct[pos]}%)")
    print(f"  {'TOTAL':<4}  ${total_sal:>6,.0f}")


In [ ]:
# Export lineup in DK upload format
if lineup is not None:
    from collections import defaultdict

    # DraftKings Classic CSV import expects exactly these 9 columns, in order.
    # Duplicate headers (RB, RB / WR, WR, WR) are valid in the DK template.
    slot_order = ["QB", "RB", "RB", "WR", "WR", "WR", "TE", "FLEX", "DST"]

    # _assign_slots (optimizer) labels each row QB/RB/WR/TE/FLEX/DST with the
    # counts that match slot_order: 1 QB, 2 RB, 3 WR, 1 TE, 1 FLEX, 1 DST.
    by_slot = defaultdict(list)
    for _, r in lineup.iterrows():
        by_slot[r["Slot"]].append(r["name"])

    # Fill each slot_order position, consuming names from the matching label.
    names_in_order, consumed = [], defaultdict(int)
    for slot in slot_order:
        bucket = by_slot.get(slot, [])
        idx = consumed[slot]
        names_in_order.append(bucket[idx] if idx < len(bucket) else "")
        consumed[slot] += 1

    dk_upload = pd.DataFrame([names_in_order], columns=slot_order)

    out_path = Path(f"dk_lineup_{SEASON}_week{WEEK:02d}.csv")
    dk_upload.to_csv(out_path, index=False)
    print(f"Lineup saved to {out_path}")
    print("Upload to DraftKings: My Lineups → Import Lineups → Upload CSV")
    print(dk_upload.to_string(index=False))


## Next Steps & Future Improvements

### Immediate
- **Review unmatched players** — any `dk_avg` players in your lineup warrant a manual
  projection check (FantasyPros consensus, snap count trends, etc.).
- **Check injury reports** — run after the official Thursday injury report drops; re-run
  the optimizer with newly-ruled-out players added to `EXCLUDED`.

### Model improvements
1. **DST projection model** — train on defensive EPA allowed, implied team total, home/away,
   and surface. Replace `dk_avg` fallback for DST.
2. **Multi-lineup GPP optimizer** — generate N lineups with ownership diversity constraints,
   forcing variation in the FLEX pick and at least one different anchor per lineup.
3. **Game-stacking correlation** — prefer combinations from the same game (e.g. QB + WR1 +
   opponent WR) which correlate positively in high-scoring contests.
4. **Salary movement signal** — compare `salary` to prior-week salary; big drops may indicate
   recency information the season-average doesn't yet capture.
5. **Integration with predict_fantasy.ipynb** — run this pipeline immediately after the
   weekly projections are generated so the process is one command end-to-end.
